# Mercedes-Benz Used Car Prices (2005–2025, USA)
## Assignment 2: Story & Insights

---

### The Story

A used Mercedes-Benz is not priced like a commodity. Its price is the result of three interacting forces:

1. **Depreciation** — how much value a car loses over time and with use
2. **Prestige** — the model hierarchy that separates a C-Class from a G-Class
3. **Usage intensity** — the independent signal that mileage carries beyond age alone

This notebook distills those three forces from 20 years of US listings into a clear, visual story — and sets the direction for a price-intelligence dashboard.

---

## Setup

In [140]:
# Plotly default renderer requires Jupyter (nbformat), which is not available in this environment.
# Therefore, I switch to the "browser" renderer to ensure figures are displayed correctly.
import plotly.io as pio
pio.renderers.default = "browser"

In [141]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import kagglehub
import os
import warnings
warnings.filterwarnings('ignore')

# ── Load data ──────────────────────────────────────────────────────────────────
path  = kagglehub.dataset_download('ibrahimshahrukh/mercedes-benz-price-dataset-2005-2025-usa')
files = os.listdir(path)
df    = pd.read_csv(os.path.join(path, files[0]))

# ── Derived columns ────────────────────────────────────────────────────────────
CURRENT_YEAR      = 2025
df['Age']         = CURRENT_YEAR - df['Year']
df['Log_Price']   = np.log1p(df['Price_USD'])
df['Miles_Per_Year'] = (df['Mileage_Miles'] / df['Age'].replace(0, np.nan)).round(0)

# ── Remove extreme outliers (< 1st and > 99th percentile on price) ─────────────
p1, p99 = df['Price_USD'].quantile([0.01, 0.99])
df_clean = df[(df['Price_USD'] >= p1) & (df['Price_USD'] <= p99)].copy()

print(f'Full dataset:    {len(df):,} rows')
print(f'Cleaned dataset: {len(df_clean):,} rows  (1st–99th percentile on price)')

Full dataset:    108 rows
Cleaned dataset: 104 rows  (1st–99th percentile on price)


---
## Part 1 — The Price Landscape

Before telling the story, we need to understand what we are predicting. Mercedes-Benz pricing spans an enormous range — from sub-$10k high-mileage C-Classes to six-figure G-Wagons and AMG supercars. The distribution is strongly right-skewed: most listings cluster between \$15k and \$60k, but the upper tail is long.

**Key takeaway:** Price must be modeled on a log scale. A $5,000 difference means something very different at the $15k end than at the $150k end.

In [142]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Price Distribution (Raw)', 'Price Distribution (Log Scale)')
)

fig.add_trace(
    go.Histogram(
        x=df_clean['Price_USD'], nbinsx=60,
        marker_color='steelblue', opacity=0.8, name='Raw Price'
    ), row=1, col=1
)
fig.add_trace(
    go.Histogram(
        x=df_clean['Log_Price'], nbinsx=60,
        marker_color='coral', opacity=0.8, name='Log Price'
    ), row=1, col=2
)

fig.update_xaxes(title_text='Price (USD)',   tickprefix='$', tickformat=',', row=1, col=1)
fig.update_xaxes(title_text='log(1 + Price)',                                row=1, col=2)
fig.update_yaxes(title_text='Count')
fig.update_layout(
    title_text='The Price Landscape',
    showlegend=False, height=420
)
fig.show()

q25, q50, q75 = df_clean['Price_USD'].quantile([0.25, 0.50, 0.75])
print(f'25th percentile: ${q25:,.0f}')
print(f'Median:          ${q50:,.0f}')
print(f'75th percentile: ${q75:,.0f}')
print(f'Skewness (raw):  {df_clean["Price_USD"].skew():.2f}')
print(f'Skewness (log):  {df_clean["Log_Price"].skew():.2f}')

25th percentile: $20,994
Median:          $33,499
75th percentile: $49,250
Skewness (raw):  1.40
Skewness (log):  -0.04


---
## Part 2 — Force 1: Depreciation

Depreciation is the single largest driver of used car prices, and Mercedes-Benz follows a well-known pattern: **steep early loss, followed by a long flat tail**. A 3-year-old vehicle retains far less of its original value than the gap between a 10-year-old and a 15-year-old car.

This non-linearity is important — it means `log(Age)` is a better feature than `Age` for any predictive model.

In [143]:
depr = (
    df_clean.groupby('Age')['Price_USD']
    .agg(median='median', q25=lambda x: x.quantile(0.25), q75=lambda x: x.quantile(0.75), count='count')
    .reset_index()
)

fig = go.Figure()

# Confidence band (IQR)
fig.add_trace(go.Scatter(
    x=pd.concat([depr['Age'], depr['Age'][::-1]]),
    y=pd.concat([depr['q75'], depr['q25'][::-1]]),
    fill='toself', fillcolor='rgba(70,130,180,0.15)',
    line=dict(color='rgba(255,255,255,0)'),
    name='IQR (25th–75th pct)', hoverinfo='skip'
))

# Median line
fig.add_trace(go.Scatter(
    x=depr['Age'], y=depr['median'],
    mode='lines+markers', line=dict(color='steelblue', width=2.5),
    marker=dict(size=6),
    name='Median Price',
    hovertemplate='Age: %{x} years<br>Median: $%{y:,.0f}<extra></extra>'
))

fig.update_layout(
    title='Depreciation Curve: Median Price vs. Vehicle Age',
    xaxis_title='Vehicle Age (Years)',
    yaxis_title='Price (USD)',
    yaxis_tickprefix='$', yaxis_tickformat=',',
    height=450, hovermode='x unified'
)


fig.show()

In [144]:
# Quantify the depreciation rate
median_new  = depr.loc[depr['Age'] == depr['Age'].min(), 'median'].values[0]
checkpoints = [1, 3, 5, 10, 15]

print(f'Reference (newest cohort): ${median_new:,.0f}\n')
print(f'{"Age":>6}  {"Median Price":>14}  {"Retained Value":>15}')
print('-' * 40)
for age in checkpoints:
    row = depr[depr['Age'] == age]
    if row.empty:
        continue
    med = row['median'].values[0]
    pct = med / median_new * 100
    print(f'{age:>6}  ${med:>13,.0f}  {pct:>14.1f}%')

Reference (newest cohort): $51,998

   Age    Median Price   Retained Value
----------------------------------------
     1  $       42,498            81.7%
     3  $       43,466            83.6%
     5  $       42,500            81.7%
    10  $       16,490            31.7%


---
## Part 3 — Force 2: Model Prestige

Not all Mercedes-Benz vehicles depreciate the same way. The brand's internal hierarchy — from entry-level A/B-Class through C/E/S-Class and up to G-Class and AMG — creates distinct price tiers that persist even as cars age.

The G-Class is the canonical example: it barely depreciates and commands a premium at every age. The C-Class, by contrast, follows aggressive depreciation typical of volume luxury sedans.

In [145]:
# Determine a sensible minimum count threshold dynamically
min_count = max(5, int(df_clean['Model_Series'].value_counts().quantile(0.4)))

series_stats = (
    df_clean.groupby('Model_Series')
    .agg(count=('Price_USD', 'count'), median_price=('Price_USD', 'median'))
    .query(f'count >= {min_count}')
    .sort_values('median_price', ascending=False)
    .head(15)
    .reset_index()
)

fig = px.bar(
    series_stats.sort_values('median_price'),
    x='median_price', y='Model_Series',
    orientation='h',
    color='median_price',
    color_continuous_scale='Blues',
    text=series_stats.sort_values('median_price')['median_price'].apply(lambda x: f'${x:,.0f}'),
    labels={'median_price': 'Median Price (USD)', 'Model_Series': ''},
    title=f'Top 15 Model Series by Median Price  (min {min_count} listings)'
)
fig.update_traces(textposition='outside')
fig.update_xaxes(tickprefix='$', tickformat=',')
fig.update_layout(height=550, coloraxis_showscale=False, showlegend=False)
fig.show()

In [146]:
# Depreciation curves for the top model series
top_series_names = series_stats.head(8)['Model_Series'].tolist()
df_top = df_clean[df_clean['Model_Series'].isin(top_series_names)]

depr_by_series = (
    df_top.groupby(['Model_Series', 'Age'])['Price_USD']
    .median().reset_index()
    .rename(columns={'Price_USD': 'median_price'})
)

fig = px.line(
    depr_by_series, x='Age', y='median_price', color='Model_Series',
    markers=True,
    labels={'Age': 'Vehicle Age (Years)', 'median_price': 'Median Price (USD)', 'Model_Series': 'Series'},
    title='Depreciation Curves by Model Series (Top 8 by median price)'
)
fig.update_yaxes(tickprefix='$', tickformat=',')
fig.update_layout(height=500, hovermode='x unified')
fig.show()

**Key takeaway:** The G-Class and top AMG models maintain value with age far better than the C/E-Class. This suggests that **model series × age interaction terms** could be valuable features in a predictive model.

---
## Part 4 — Force 3: Usage Intensity (Mileage)

Age tells you *how old* a car is; mileage tells you *how hard it was driven*. These are related but not identical. A 10-year-old car with 30,000 miles is priced very differently from a 10-year-old car with 150,000 miles.

The following analysis shows that mileage carries **independent predictive power** beyond age — confirming that both must be included in any model.

In [147]:
# 2D density: Age vs Mileage, colored by median price
# Bin into age groups and mileage buckets
df_clean['Age_Group']     = pd.cut(df_clean['Age'],          bins=[0,2,5,10,15,25], labels=['0-2','3-5','6-10','11-15','16+'])
df_clean['Mileage_Group'] = pd.cut(df_clean['Mileage_Miles'], bins=[0,25000,50000,75000,100000,150000,300000],
                                    labels=['<25k','25-50k','50-75k','75-100k','100-150k','150k+'])

heat = (
    df_clean.groupby(['Age_Group', 'Mileage_Group'], observed=True)['Price_USD']
    .median().reset_index()
    .pivot(index='Mileage_Group', columns='Age_Group', values='Price_USD')
)

fig = px.imshow(
    heat,
    color_continuous_scale='RdYlGn',
    labels=dict(x='Vehicle Age Group', y='Mileage Group', color='Median Price'),
    title='Median Price Heatmap: Age Group × Mileage Group',
    text_auto='.0f'
)
fig.update_coloraxes(colorbar_tickprefix='$')
fig.update_layout(height=420)
fig.show()

In [148]:
# Mileage vs price, faceted by age group
fig = px.scatter(
    df_clean[df_clean['Age_Group'].notna()],
    x='Mileage_Miles',
    y='Price_USD',
    facet_col='Age_Group',
    facet_col_wrap=3,
    opacity=0.25,
    trendline='lowess',
    labels={
        'Mileage_Miles': 'Mileage',
        'Price_USD': 'Price (USD)',
        'Age_Group': 'Age Group'
    },
    title='Price vs Mileage Within Each Age Group'
)

fig.update_yaxes(tickprefix='$', tickformat=',')
fig.update_xaxes(tickformat=',')
fig.update_layout(height=600)

fig.show()

Within every age group, mileage still has a clear downward slope on price. The heatmap makes this especially clear: the bottom-right cell (oldest AND most miles) is always the lowest price, while the top-left (newest AND fewest miles) is always the highest.

---
## Part 5 — Body Type: A Secondary but Meaningful Signal

Body type is partly a proxy for model series, but it carries independent signal. SUVs and convertibles command a consistent premium over sedans and hatchbacks at equivalent age and mileage levels. This reflects both market demand and the underlying models that occupy each body type.

In [149]:
body_order = df_clean.groupby('Body_Type')['Price_USD'].median().sort_values(ascending=False).index.tolist()

fig = px.box(
    df_clean,
    x='Body_Type', y='Price_USD',
    category_orders={'Body_Type': body_order},
    color='Body_Type',
    labels={'Price_USD': 'Price (USD)', 'Body_Type': ''},
    title='Price Distribution by Body Type (sorted by median)'
)
fig.update_yaxes(tickprefix='$', tickformat=',')
fig.update_layout(height=480, showlegend=False)
fig.show()

In [150]:
# Volume vs median price bubble chart
body_stats = (
    df_clean.groupby('Body_Type')
    .agg(count=('Price_USD','count'), median_price=('Price_USD','median'), mean_age=('Age','mean'))
    .reset_index()
)

fig = px.scatter(
    body_stats,
    x='mean_age', y='median_price',
    size='count', color='Body_Type', text='Body_Type',
    labels={'mean_age': 'Mean Vehicle Age (Years)', 'median_price': 'Median Price (USD)', 'count': 'Listings'},
    title='Body Type: Market Volume vs Median Price (bubble = listing count)'
)
fig.update_traces(textposition='top center')
fig.update_yaxes(tickprefix='$', tickformat=',')
fig.update_layout(height=480, showlegend=False)
fig.show()

---
## Part 6 — Market Segmentation: The Full Picture

Combining age, mileage, and model series in one view reveals the three distinct market segments that will anchor the dashboard:

- **Premium segment** — newer, lower mileage, G-Class / AMG / S-Class
- **Mid-market** — 3–10 year old E/C-Class, moderate mileage
- **Budget segment** — older, higher mileage, A/B/C-Class

In [151]:
# Interactive scatter: Mileage vs Price, colored by Age, filterable by Model Series
df_plot = df_clean[df_clean['Model_Series'].isin(top_series_names)].copy()

fig = px.scatter(
    df_plot,
    x='Mileage_Miles', y='Price_USD',
    color='Age',
    facet_col='Model_Series', facet_col_wrap=4,
    color_continuous_scale='RdYlGn_r',
    opacity=0.5,
    labels={'Mileage_Miles': 'Mileage', 'Price_USD': 'Price', 'Age': 'Age (yrs)'},
    title='Price vs Mileage by Model Series, colored by Age'
)
fig.update_yaxes(tickprefix='$', tickformat=',', matches=None)
fig.update_xaxes(tickformat=',')
fig.update_layout(height=700)
fig.show()

In [152]:
# Treemap: share of listings by Body Type → Model Series
tree = (
    df_clean.groupby(['Body_Type', 'Model_Series'])
    .agg(count=('Price_USD','count'), median_price=('Price_USD','median'))
    .reset_index()
    .query('count >= 5')
)

fig = px.treemap(
    tree,
    path=['Body_Type', 'Model_Series'],
    values='count',
    color='median_price',
    color_continuous_scale='Blues',
    labels={'median_price': 'Median Price', 'count': 'Listings'},
    title='Listing Volume by Body Type → Model Series  (color = median price)'
)
fig.update_coloraxes(colorbar_tickprefix='$', colorbar_tickformat=',')
fig.update_layout(height=550)
fig.show()

---
## Part 7 — Identifying Underpriced Listings

One practical application of this dataset is **detecting listings priced significantly below the market rate for their cohort** — same model series, similar age, similar mileage. This is a direct use case for a dashboard: a buyer filtering for deals.

We approximate this by computing, for each listing, the deviation from the median price of its (Model_Series, Age_Group, Mileage_Group) cohort.

In [153]:
cohort_median = (
    df_clean.groupby(['Model_Series', 'Age_Group', 'Mileage_Group'], observed=True)['Price_USD']
    .median()
    .reset_index()
    .rename(columns={'Price_USD': 'cohort_median'})
)

df_scored = df_clean.merge(cohort_median, on=['Model_Series', 'Age_Group', 'Mileage_Group'], how='left')
df_scored['Price_vs_Cohort_Pct'] = (df_scored['Price_USD'] - df_scored['cohort_median']) / df_scored['cohort_median'] * 100

# Show distribution of the discount/premium
fig = px.histogram(
    df_scored.dropna(subset=['Price_vs_Cohort_Pct']),
    x='Price_vs_Cohort_Pct', nbins=80,
    color_discrete_sequence=['steelblue'],
    labels={'Price_vs_Cohort_Pct': 'Price vs Cohort Median (%)'},
    title='Distribution of Listing Price Relative to Cohort Median'
)
fig.add_vline(x=0, line_dash='dash', line_color='red', annotation_text='Cohort median')
fig.add_vline(x=-20, line_dash='dot', line_color='green', annotation_text='-20% threshold')
fig.update_layout(height=420)
fig.show()

In [154]:
# Top underpriced listings (>20% below cohort median)
deals = (
    df_scored[df_scored['Price_vs_Cohort_Pct'] < -20]
    [['Year', 'Model_Series', 'Body_Type', 'Mileage_Miles', 'Price_USD', 'cohort_median', 'Price_vs_Cohort_Pct']]
    .sort_values('Price_vs_Cohort_Pct')
    .head(20)
    .reset_index(drop=True)
)

fig = px.scatter(
    deals,
    x='Price_vs_Cohort_Pct', y='Price_USD',
    color='Model_Series', size='Mileage_Miles',
    hover_data=['Year', 'Body_Type', 'Mileage_Miles', 'cohort_median'],
    labels={
        'Price_vs_Cohort_Pct': 'Discount vs Cohort (%)',
        'Price_USD': 'Listed Price (USD)'
    },
    title='Potentially Underpriced Listings (>20% below cohort median)'
)
fig.update_yaxes(tickprefix='$', tickformat=',')
fig.update_layout(height=480)
fig.show()

---
## Summary: The Three-Force Model

The story of Mercedes-Benz used car prices in the US reduces to three forces:

| Force | Key Variables | Strength |
|---|---|---|
| **Depreciation** | Age, log(Age) | Strong, non-linear |
| **Model Prestige** | Model_Series, Body_Type | Strong, tier-based |
| **Usage Intensity** | Mileage, Miles_Per_Year | Moderate, independent of age |

These three forces interact: a high-mileage G-Class at 10 years old still commands more than a low-mileage C-Class at 5 years old. Model prestige can dominate both depreciation and mileage at the extremes.

---

## Dashboard Plan

The final dashboard will have three panels:

1. **Market Overview** — price distribution by model series and body type; treemap of volume + pricing. Lets a user see "where does my car sit in the market?"

2. **Depreciation Explorer** — interactive depreciation curves per model series, with sliders for mileage range. Answers: "how fast is my model losing value?"

3. **Deal Finder** — cohort-relative pricing scatter. Filters by model, age, and mileage to surface listings priced below their cohort median. Answers: "is this listing a good deal?"

The primary model underlying the dashboard will be a gradient boosting regressor trained on `log(Price_USD)` with features: `Age`, `log(Age)`, `Mileage_Miles`, `Miles_Per_Year`, `Model_Series` (encoded), and `Body_Type` (encoded).